[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/37_lora_solution.ipynb)

# 🟡 Solution: LoRA (Low-Rank Adaptation)

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `37_lora.ipynb` first.

---
Implement a **LoRA**-adapted linear layer as an `nnx.Module`.

$$h = xW + \frac{\alpha}{r}\,(xA)B$$

where $W \in \mathbb{R}^{d_{in}\times d_{out}}$ is **frozen**,
$A \in \mathbb{R}^{d_{in}\times r}$ and $B \in \mathbb{R}^{r\times d_{out}}$ are
trainable, and $r \ll \min(d_{in}, d_{out})$.

### Signature
```python
class LoRALinear(nnx.Module):
    def __init__(self, din, dout, rank, alpha=1.0, *, rngs: nnx.Rngs):
        ...
    def __call__(self, x):
        ...  # (..., din) -> (..., dout)
```

### Rules
- The base weight must **not** be an `nnx.Param` — use a plain `nnx.Variable`,
  so `nnx.split(model, nnx.Param, ...)` sees only the adapter
- `A`: random init (scaled normal is fine). `B`: **zeros**
- Keep `(x @ A) @ B` factored; never form `A @ B`
- Scale by `alpha / rank`
- No bias term

### Why B must be zero
At initialization $BA = 0$, so $h = xW$ exactly — the adapted model is
**identical** to the base model. Training therefore starts from the pretrained
function rather than from a randomly perturbed one, which is what makes LoRA
stable at high learning rates.

Both matrices zero would be broken instead: $\partial \mathcal{L}/\partial A
\propto B^\top = 0$ and $\partial\mathcal{L}/\partial B \propto (xA)^\top = 0$,
so nothing ever moves. You need exactly one of them zero — the product vanishes
but the gradients do not. (By symmetry, random-$B$/zero-$A$ works too;
zero-$B$ is the convention.)

### The arithmetic that makes it worth it
For $d_{in}=d_{out}=4096$: full fine-tuning trains $16.8$M parameters per
matrix. LoRA at $r=8$ trains $2 \cdot 4096 \cdot 8 = 65$K — **0.39%**.

The memory win is bigger than the parameter count suggests, because Adam keeps
*two* fp32 moments per trainable parameter ([[adam]]). Optimizer state drops by
the same 256x, and that — not the parameter count — is usually what decides
whether a model fits on your GPU.

### Why $\alpha/r$ and not just $\alpha$
The scale keeps the update magnitude roughly constant as you change $r$, so
retuning the rank does not force you to retune the learning rate. In practice
people fix $\alpha$ (often $2r$) and sweep $r$.

### The deployment property
After training, $W' = W + \frac{\alpha}{r}AB$ can be folded into the base weight
once, giving a plain linear layer with **zero** added inference latency —
unlike adapter layers, which add depth to the forward pass. And since the base
weight is untouched, many task-specific LoRAs can be swapped against one shared
frozen model.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class LoRALinear(nnx.Module):
    """Linear layer with a frozen base weight and a trainable low-rank adapter."""

    def __init__(self, din: int, dout: int, rank: int, alpha: float = 1.0,
                 *, rngs: nnx.Rngs):
        self.din, self.dout, self.rank = din, dout, rank
        self.scaling = alpha / rank

        # Plain nnx.Variable, NOT nnx.Param — this is what "frozen" means here:
        # an nnx.Param filter will not collect it, so no gradient is produced.
        key_w, key_a = jax.random.split(rngs.params(), 2)
        self.W = nnx.Variable(
            jax.random.normal(key_w, (din, dout)) * (1.0 / jnp.sqrt(din))
        )

        # A random, B zero -> the adapter contributes exactly nothing at init,
        # while both still receive gradient on the first step.
        self.A = nnx.Param(jax.random.normal(key_a, (din, rank)) * 0.01)
        self.B = nnx.Param(jnp.zeros((rank, dout)))

    def __call__(self, x):
        base = x @ self.W.value
        # Kept factored: (..., din) @ (din, r) @ (r, dout). Forming A @ B first
        # would build the full (din, dout) matrix LoRA exists to avoid.
        delta = (x @ self.A.value) @ self.B.value
        return base + self.scaling * delta

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

layer = LoRALinear(64, 64, rank=4, alpha=8.0, rngs=nnx.Rngs(0))
x = jax.random.normal(jax.random.key(1), (2, 64))

print("adapter output at init:", float(jnp.abs(layer(x) - x @ layer.W.value).max()))

params = nnx.state(layer, nnx.Param)
n = sum(p.size for p in jax.tree.leaves(params))
print(f"trainable params: {n}  (full weight would be {64 * 64})")
print(f"                  {100 * n / (64 * 64):.1f}% of full fine-tuning")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("lora")